## Read from textgrid

In [ ]:
from adapt_eeg.auto_annotation.syllable_from_textgrid import load_textgrid_syllable

syllablized_audio = load_textgrid_syllable(
    "../data/raw-audio/1. Sailing_to_byzantium_simona_final.wav",
    "../data/text-grid/01_Sailing_to_byzantium_simona_final1_Textgrid.TextGrid",
    intensity_floor_hz=50,
    pitch_timestep_s=0.01,
    pitch_range_hz=(75, 500),
    silence_threshold_relative_db=-25,
    syllable_pitch_percentile=90,
    min_nucleus_prominence_db=2,
)

syllablized_audio = syllablized_audio[2500:]  # cut off the intro

In [ ]:


syllables = [s for s in syllablized_audio.syllables]
syllable_times = [s.nucleus for s in syllablized_audio.syllables]

len(syllable_times)

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

durations = [s.duration_s for s in syllables]

q25, q50, q75 = np.percentile(durations, [25, 50, 75])
q0, q100 = min(durations), max(durations)

print(q0, q25, q50, q75, q100)

plt.hist([s.duration_s for s in syllables if s.duration_s <= 0.1 ])
plt.show()

In [ ]:
short_syllables = [s for s in syllables if s.duration_s <= 0.1]

In [ ]:
short_syllables[0]

In [ ]:
syllablized_audio.syllables[7].start, syllablized_audio.syllables[7].end

In [ ]:
from adapt_eeg.utils.audio_stress_visualization import waveform_render
from IPython.display import HTML, display

def waveform_player(audio,
    timestamps,
    window_seconds=20,
    height=160,
):
    html = waveform_render(audio, timestamps, window_seconds, height)
    display(HTML(html))

In [ ]:

waveform_player(syllablized_audio.audio, syllable_times)

## detect stress

In [ ]:
from adapt_eeg.auto_annotation.classes import StressIdentifyParams
from adapt_eeg.auto_annotation.exceptions import InsufficientEligibleSyllablesError
from adapt_eeg.auto_annotation.stress_identifier import detect_stressed_syllables

params = StressIdentifyParams(
    pitch_range_hz=(75.0, 500.0),
    prominence_neighbor_half_window_s=0.05,
    prominence_min_nucleus_distance_s=0.05,
    min_pitch_prominence_hz=0.52,  # 2
    min_intensity_prominence_db=0.28,  # 5
    duration_neighbor_radius=3,
    stress_weights=(5, 3, 2),
    syllables_per_line=10,
    stressed_syllables_per_line=5
)

err_syls = None
try:
    stresses = detect_stressed_syllables(syllablized_audio, params)
    stress_times = [stress.nucleus for stress in stresses]
    print(len(stress_times))
except InsufficientEligibleSyllablesError as e:
    print(e)
    err_syls = e.syllables
    for evi, syl in zip(e.evidence, e.syllables):
        print(evi, syl.transcription)


In [ ]:
# from adapt_eeg.auto_annotation.stress_identifier import (
#     _normalize_weights,
#     calculate_stress_evidence,
#     calculate_stress_scores,
#     classify_stresses,
#     normalize_stress_evidence,
# )
# from adapt_eeg.auto_annotation.syllable_identifier import SyllablizedAudio


# syllables = syllablized_audio.syllables


# evidence = calculate_stress_evidence(
#     syllablized_audio,
#     params.prominence_neighbor_half_window_s,
#     params.prominence_min_nucleus_distance_s,
#     duration_neighbor_radius=params.duration_neighbor_radius,
#     pitch_floor_hz=params.pitch_range_hz[0],
# )

# normalized_evidence = normalize_stress_evidence(evidence)

# weights = _normalize_weights(params.stress_weights)

# scores = calculate_stress_scores(
#     normalized_evidence,
#     weights,
# )

# stress_flags = classify_stresses(
#     evidence,
#     scores,
#     params.min_pitch_prominence_hz,
#     params.min_intensity_prominence_db,
#     params.stress_score_threshold,
# )

# stresses = [syllable for syllable, stressed in zip(syllables, stress_flags) if stressed]
# stress_times = [stress.nucleus for stress in stresses]
# len(stress_times)

In [ ]:
waveform_player(syllablized_audio.audio, stress_times)

In [ ]:
for i, (syllable, syllable_time, score, flag) in enumerate(zip(syllables, evidence, scores, stress_flags)):
    print(syllable.transcription, syllable_time, score, flag)
    if i > 20:
        break

In [ ]:
import numpy as np

pitch_prom = np.array([e[0] for e in evidence], dtype=float)
intensity_prom = np.array([e[1] for e in evidence], dtype=float)

pitch_prom = pitch_prom[np.isfinite(pitch_prom)]
intensity_prom = intensity_prom[np.isfinite(intensity_prom)]

pitch_80_threshold = np.percentile(pitch_prom, 20)
intensity_80_threshold = np.percentile(intensity_prom, 20)
score_60 = np.percentile(scores, 40)

print("pitch threshold for ~80% pass:", pitch_80_threshold)
print("intensity threshold for ~80% pass:", intensity_80_threshold)
print("score threshold for ~80% pass:", score_60)

In [ ]:
pitch_pass = [e[0] >= params.min_pitch_prominence_hz for e in evidence]

intensity_pass = [e[1] >= params.min_intensity_prominence_db for e in evidence]

score_pass = [score >= params.stress_score_threshold for score in scores]

n = len(syllables)

print("pitch:", sum(pitch_pass), f"({100 * sum(pitch_pass) / n:.1f}%)")

print("intensity:", sum(intensity_pass), f"({100 * sum(intensity_pass) / n:.1f}%)")

print(
    "pitch + intensity:",
    sum(p and i for p, i in zip(pitch_pass, intensity_pass)),
    f"({100 * sum(p and i for p, i in zip(pitch_pass, intensity_pass)) / n:.1f}%)",
)

print("score:", sum(score_pass), f"({100 * sum(score_pass) / n:.1f}%)")

all = sum(
        p and i and s
        for p, i, s in zip(
            pitch_pass,
            intensity_pass,
            score_pass,
        )
    )
print(
    "all:",
    all,
    f"({100 * all / n:.1f}%)",
)